In [1]:
import os
import cv2
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
import json
import shutil

In [2]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.9.0+cpu
CUDA available: False


In [3]:
os.makedirs('datasets', exist_ok=True)
os.makedirs('runs', exist_ok=True)
os.makedirs('predictions/images', exist_ok=True)
os.makedirs('predictions/labels', exist_ok=True)
os.makedirs('reports', exist_ok=True)


In [4]:
dataset_config = {
    'path': './datasets/space_station',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 7,
    'names': [
        'OxygenTank',
        'NitrogenTank', 
        'FirstAidBox',
        'FireAlarm',
        'SafetySwitchPanel',
        'EmergencyPhone',
        'FireExtinguisher'
    ]
}

with open('datasets/space_station.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

print("Dataset configuration created!")


Dataset configuration created!


In [5]:
class DataVisualizer:
    def __init__(self, dataset_path):
        self.dataset_path = Path(dataset_path)
        self.classes = dataset_config['names']
        self.colors = plt.cm.Set3(np.linspace(0, 1, len(self.classes)))
    
    def plot_sample_images(self, split='train', num_samples=8):
        """Plot sample images with bounding boxes"""
        images_dir = self.dataset_path / split / 'images'
        labels_dir = self.dataset_path / split / 'labels'
        
        image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
        np.random.shuffle(image_files)
        
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        axes = axes.ravel()
        
        for idx, img_path in enumerate(image_files[:8]):
            # Load image
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Load labels
            label_path = labels_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                
                h, w = img.shape[:2]
                for line in lines:
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Convert normalized coordinates to pixels
                    x_center *= w
                    y_center *= h
                    width *= w
                    height *= h
                    
                    x1 = int(x_center - width/2)
                    y1 = int(y_center - height/2)
                    x2 = int(x_center + width/2)
                    y2 = int(y_center + height/2)
                    
                    # Draw bounding box
                    color = [int(c * 255) for c in self.colors[int(class_id)][:3]]
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    cv2.putText(img, self.classes[int(class_id)], (x1, y1-10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            axes[idx].imshow(img)
            axes[idx].set_title(f'Sample {idx+1}')
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.savefig('reports/sample_images.jpg', dpi=300, bbox_inches='tight')
        plt.show()
    

In [6]:
def analyze_class_distribution(self):
        """Analyze class distribution across splits"""
        splits = ['train', 'val', 'test']
        class_counts = {split: {cls: 0 for cls in self.classes} for split in splits}
        
        for split in splits:
            labels_dir = self.dataset_path / split / 'labels'
            if not labels_dir.exists():
                continue
                
            for label_file in labels_dir.glob('*.txt'):
                with open(label_file, 'r') as f:
                    lines = f.readlines()
                for line in lines:
                    class_id = int(line.strip().split()[0])
                    class_counts[split][self.classes[class_id]] += 1